# Mini-CLIP — The Library Version

Three parts, honestly framed: (1) the symmetric InfoNCE verified against a slow per-pair loop; (2) our result placed against real CLIP's, with the scale ledger stated plainly; (3) the real-CLIP translation — shown, not run.

In [1]:
import numpy as np

def softmax(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=-1, keepdims=True)

# (1) the symmetric InfoNCE, vectorized vs the slow honest per-pair loop
rs = np.random.default_rng(0)
m, E = 8, 16
Ei = rs.normal(0, 1, (m, E)); Ei /= np.linalg.norm(Ei, axis=1, keepdims=True)
Et = rs.normal(0, 1, (m, E)); Et /= np.linalg.norm(Et, axis=1, keepdims=True)
s = 10.0
logits = s * Ei @ Et.T
Pi, Pt = softmax(logits), softmax(logits.T)
L_vec = 0.5 * (-np.mean(np.log(np.diag(Pi))) - np.mean(np.log(np.diag(Pt))))

L_loop = 0.0
for i in range(m):                                  # each image argues against every caption...
    row = np.exp(logits[i] - logits[i].max())
    L_loop += -np.log(row[i] / row.sum())
for j in range(m):                                  # ...and each caption against every image
    col = np.exp(logits[:, j] - logits[:, j].max())
    L_loop += -np.log(col[j] / col.sum())
L_loop /= (2 * m)
print(f"vectorized InfoNCE {L_vec:.10f} vs per-pair loop {L_loop:.10f} — difference {abs(L_vec-L_loop):.2e}")

vectorized InfoNCE 3.4264099007 vs per-pair loop 3.4264099007 — difference 4.44e-16


In [2]:
# (2) the scale ledger, stated plainly
print("OURS vs CLIP (2021) — same recipe, different planets:")
print(f"{'':26s} {'mini-CLIP':>18s} {'CLIP ViT-L':>22s}")
rows = [("pairs", "2,700", "400,000,000"),
        ("batch size", "128", "32,768"),
        ("parameters", "19,297", "~430,000,000"),
        ("zero-shot (seen classes)", "82% (8-way)", "76% (ImageNet, 1000-way)"),
        ("novel compositions", "size yes / shape no", "strong, still imperfect")]
for r in rows: print(f"{r[0]:26s} {r[1]:>18s} {r[2]:>22s}")
print()
print("The recipe is identical to the line; what scale buys is the SPACE's richness — enough")
print("co-variation that attributes factorize (our size did; our shape needed more). Even at")
print("400M pairs, compositional and fine-grained gaps remain CLIP's known weak spots: the")
print("same failure we measured, pushed further out, never fully gone.")

OURS vs CLIP (2021) — same recipe, different planets:
                                    mini-CLIP             CLIP ViT-L
pairs                                   2,700            400,000,000
batch size                                128                 32,768
parameters                             19,297           ~430,000,000
zero-shot (seen classes)          82% (8-way) 76% (ImageNet, 1000-way)
novel compositions         size yes / shape no strong, still imperfect

The recipe is identical to the line; what scale buys is the SPACE's richness — enough
co-variation that attributes factorize (our size did; our shape needed more). Even at
400M pairs, compositional and fine-grained gaps remain CLIP's known weak spots: the
same failure we measured, pushed further out, never fully gone.


### (3) The real-CLIP translation — read it; you have built every line

```python
import torch, torch.nn.functional as F
# ... image_encoder = a ViT (lesson 15); text_encoder = a transformer (lesson 14) ...

image_features = F.normalize(image_encoder(images) @ W_i, dim=-1)   # Block 4
text_features  = F.normalize(text_encoder(tokens) @ W_t, dim=-1)
logits = logit_scale.exp() * image_features @ text_features.T        # Block 5
labels = torch.arange(len(images))
loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

# zero-shot (openai/CLIP or open_clip, pretrained):
#   texts = clip.tokenize([f"a photo of a {c}" for c in classes])
#   pred = (model.encode_image(img) @ model.encode_text(texts).T).argmax()
```

Five lines of loss — the exact five you derived, checked, and trained. The 400M pairs are the part you can't type.